In [ ]:
import requests
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import col, round as spark_round

# notebookutils.credentials.getToken hands back the *user's* AAD token scoped
# for the resource you ask for. No service principal, no secrets in the
# notebook. Works for both interactive and pipeline runs.
FABRIC_TOK = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
H = {"Authorization": f"Bearer {FABRIC_TOK}"}

WORKSPACE_ID = spark.conf.get("trident.workspace.id")
BASE = "https://api.fabric.microsoft.com/v1"

print(f"Workspace: {WORKSPACE_ID}")

In [ ]:
# ----------------------------------------------------------------------------
# Parameters -- tweak these before running
# ----------------------------------------------------------------------------

# How many days of history to pull. 7 is usually plenty.
LOOKBACK_DAYS = 7

# Set to None to scan every notebook in the workspace, or pass an allow-list
# of display names to scope it (faster, less API traffic).
NOTEBOOK_FILTER = None
# Example:
# NOTEBOOK_FILTER = [
#     "00_run_log_init",
#     "01_synthea_generate",
#     "02_bronze_to_silver",
#     "03_silver_to_gold",
# ]

# Set True to persist a snapshot of this report to a Delta table in gold.
# Default False so you can preview before committing.
PERSIST_TO_DELTA = False
PERSIST_TABLE = "agent.notebook_timing_report"

In [ ]:
# ----------------------------------------------------------------------------
# 1. Discover notebooks in the workspace
# ----------------------------------------------------------------------------

r = requests.get(f"{BASE}/workspaces/{WORKSPACE_ID}/notebooks", headers=H, timeout=60)
r.raise_for_status()
all_notebooks = r.json().get("value", [])

if NOTEBOOK_FILTER:
    notebooks = [n for n in all_notebooks if n["displayName"] in NOTEBOOK_FILTER]
else:
    notebooks = all_notebooks

print(f"Scanning {len(notebooks)} notebook(s):")
for n in notebooks:
    print(f"  - {n['displayName']:<40} {n['id']}")

In [ ]:
# ----------------------------------------------------------------------------
# 2. For each notebook, pull job instances within the lookback window
# ----------------------------------------------------------------------------

def _parse_iso(s):
    if not s:
        return None
    return datetime.fromisoformat(s.replace("Z", "+00:00"))

cutoff = datetime.now(timezone.utc).timestamp() - (LOOKBACK_DAYS * 86400)
instances = []  # one record per notebook job instance

for nb in notebooks:
    nb_id, nb_name = nb["id"], nb["displayName"]
    url = f"{BASE}/workspaces/{WORKSPACE_ID}/items/{nb_id}/jobs/instances"
    page = 0
    while url and page < 20:  # safety cap
        page += 1
        r = requests.get(url, headers=H, timeout=60)
        if r.status_code == 429:
            import time; time.sleep(int(r.headers.get("Retry-After", "10"))); continue
        r.raise_for_status()
        body = r.json()
        for ji in body.get("value", []):
            start = _parse_iso(ji.get("startTimeUtc"))
            if start and start.timestamp() < cutoff:
                continue
            instances.append({
                "notebook":         nb_name,
                "notebook_id":      nb_id,
                "job_instance_id":  ji["id"],
                "status":           ji.get("status"),
                "invoke_type":      ji.get("invokeType"),     # "Manual" | "Scheduled" | "OnDemand"
                "root_activity_id": ji.get("rootActivityId"), # == Livy operation ID for Spark
                "activity_start":   start,
                "activity_end":     _parse_iso(ji.get("endTimeUtc")),
            })
        url = body.get("continuationUri")

print(f"Found {len(instances)} job instance(s) in the last {LOOKBACK_DAYS} day(s).")

In [ ]:
# ----------------------------------------------------------------------------
# 3. Pull Spark application detail for each instance (keyed on Livy ID)
# ----------------------------------------------------------------------------
#
# Fabric exposes the Spark application list per workspace:
#   GET /v1/workspaces/{ws}/spark/applications
# Each entry includes livyId, submittedAt, endedAt, totalDuration, queueWait.
# We index by livyId then join back to the notebook job instances.

spark_apps = {}
url = f"{BASE}/workspaces/{WORKSPACE_ID}/spark/applications?$top=200"
page = 0
while url and page < 50:
    page += 1
    r = requests.get(url, headers=H, timeout=60)
    if r.status_code == 429:
        import time; time.sleep(int(r.headers.get("Retry-After", "10"))); continue
    if r.status_code == 404:
        print("Spark monitoring API not available in this workspace -- "
              "pool_startup_sec will be null. Activity duration still reported.")
        break
    r.raise_for_status()
    body = r.json()
    for a in body.get("value", []):
        livy = a.get("livyId") or a.get("operationId")
        if livy:
            spark_apps[livy] = a
    url = body.get("continuationUri")

print(f"Indexed {len(spark_apps)} Spark application(s).")

In [ ]:
# ----------------------------------------------------------------------------
# 4. Join + compute the timing breakdown per instance
# ----------------------------------------------------------------------------

def _secs(a, b):
    if a is None or b is None:
        return None
    return (b - a).total_seconds()

rows = []
for j in instances:
    s = spark_apps.get(j["root_activity_id"])
    sp_submit = _parse_iso(s.get("submittedAt")) if s else None
    sp_end    = _parse_iso(s.get("endedAt"))     if s else None
    queue_wait = s.get("queueWaitInSeconds") if s else None

    activity_dur = _secs(j["activity_start"], j["activity_end"])
    spark_dur    = _secs(sp_submit, sp_end)
    pool_startup = _secs(j["activity_start"], sp_submit)
    teardown     = _secs(sp_end, j["activity_end"])

    rows.append(Row(
        notebook         = j["notebook"],
        status           = j["status"],
        invoke_type      = j["invoke_type"],
        activity_start   = j["activity_start"],
        activity_end     = j["activity_end"],
        activity_sec     = activity_dur,
        capacity_wait_sec= float(queue_wait) if queue_wait is not None else None,
        pool_startup_sec = pool_startup,
        spark_app_sec    = spark_dur,
        teardown_sec     = teardown,
        job_instance_id  = j["job_instance_id"],
        livy_id          = j["root_activity_id"],
    ))

df = spark.createDataFrame(rows)

# Round the numeric columns for readability.
for c in ("activity_sec", "capacity_wait_sec", "pool_startup_sec",
          "spark_app_sec", "teardown_sec"):
    df = df.withColumn(c, spark_round(col(c), 1))

df = df.orderBy(col("activity_start").desc())
df.cache()

print(f"Computed timing for {df.count()} run(s).")
display(df)

In [ ]:
# ----------------------------------------------------------------------------
# 5. Quick aggregate -- average pool startup per notebook
# ----------------------------------------------------------------------------

from pyspark.sql.functions import avg, max as smax, min as smin, count

summary = (
    df.filter(col("status") == "Completed")
      .groupBy("notebook")
      .agg(
          count("*").alias("runs"),
          spark_round(avg("pool_startup_sec"), 1).alias("avg_pool_startup_sec"),
          spark_round(smin("pool_startup_sec"), 1).alias("min_pool_startup_sec"),
          spark_round(smax("pool_startup_sec"), 1).alias("max_pool_startup_sec"),
          spark_round(avg("spark_app_sec"),    1).alias("avg_spark_app_sec"),
          spark_round(avg("activity_sec"),     1).alias("avg_activity_sec"),
      )
      .orderBy(col("avg_pool_startup_sec").desc_nulls_last())
)
display(summary)

In [ ]:
# ----------------------------------------------------------------------------
# 6. Optional: persist this snapshot to a Delta table in gold
# ----------------------------------------------------------------------------

if PERSIST_TO_DELTA:
    from pyspark.sql.functions import current_timestamp, lit
    snap = df.withColumn("collected_at", current_timestamp()) \
             .withColumn("workspace_id", lit(WORKSPACE_ID))
    (snap.write.mode("append")
         .option("mergeSchema", "true")
         .saveAsTable(PERSIST_TABLE))
    print(f"Appended {snap.count()} row(s) to {PERSIST_TABLE}.")
else:
    print("PERSIST_TO_DELTA = False -- skipping write.")